In [57]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
from datetime import datetime, timedelta
# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

In [58]:
hormones_selfrep = pd.read_csv(r'D:\MajorProject\mcphases\hormones_and_selfreport.csv')
resting_hr = pd.read_csv(r'D:\MajorProject\mcphases\resting_heart_rate.csv')
wrist_temp = pd.read_csv(r'D:\MajorProject\mcphases\wrist_temperature.csv')

In [59]:
hormones_selfrep.isnull().mean().sort_values(ascending=False)*100

pdg               67.061318
flow_volume       43.647288
flow_color        43.558933
moodswing         41.332391
indigestion       41.244036
stress            41.208694
foodcravings      41.208694
cramps            41.208694
sorebreasts       41.208694
bloating          41.191023
headaches         41.191023
sleepissue        41.173352
exerciselevel     41.155681
appetite          41.155681
fatigue           41.138010
estrogen           5.672380
lh                 5.654709
phase              0.017671
is_weekend         0.000000
day_in_study       0.000000
id                 0.000000
study_interval     0.000000
dtype: float64

In [60]:
hormones_selfrep['phase'].unique()

array(['Follicular', 'Fertility', 'Luteal', 'Menstrual', nan],
      dtype=object)

In [61]:
print(hormones_selfrep['phase'].isna().sum())
print(hormones_selfrep['phase'].value_counts(dropna=False))

1
phase
Luteal        1912
Follicular    1386
Fertility     1281
Menstrual     1079
NaN              1
Name: count, dtype: int64


In [62]:
hormones_selfrep['phase'] = hormones_selfrep['phase'].fillna(method='ffill')  

C:\Users\iqra khan\AppData\Local\Temp\ipykernel_29008\2835141073.py:1: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  hormones_selfrep['phase'] = hormones_selfrep['phase'].fillna(method='ffill')


In [63]:
hormones_selfrep['is_bleeding'] = hormones_selfrep['phase'].apply(
    lambda x: 1 if x == 'Menstrual' else 0
)

created a *is_bleeding* column using phase column this will determine flow volume which will determine start date of period in explore_mcphases


In [64]:
flow_clr_map = {
    "Bright Red": "Fresh",
    "Pink": "Fresh",
    "Dark Brown / Dark Red": "Old",
    "Black": "Old",
    "Grey": "Abnormal",
    "Orange": "Abnormal",
    "Yellow": "Abnormal",
    "Other": "Abnormal",
    "Not at all": "No Flow"
}
hormones_selfrep['flow_grp'] = hormones_selfrep['flow_color'].map(flow_clr_map)

hormones_selfrep = pd.get_dummies(hormones_selfrep, columns = ['flow_grp'])

In [65]:
hormones_selfrep['flow_volume'].unique()

array(['Not at all', nan, 'Very Heavy', 'Somewhat Heavy', 'Moderate',
       'Light', 'Spotting / Very Light', 'Heavy', 'Somewhat Light'],
      dtype=object)

In [88]:
hormones_selfrep['flow_volume'].isna().sum()

np.int64(2470)

##### fill flow volume

In [66]:
# Step 1: Map categorical flow volume to numbers
flow_volume_map = {
    'Not at all':            0,
    'Spotting / Very Light': 1,
    'Light':                 2,
    'Somewhat Light':        2.5,
    'Moderate':              3,
    'Somewhat Heavy':        3.5,
    'Heavy':                 4,
    'Very Heavy':            5
}

hormones_selfrep['flow_volume_numeric'] = hormones_selfrep['flow_volume'].map(flow_volume_map)

In [ ]:
# Calculate period_start per user
hormones_selfrep['period_start'] = (
    (hormones_selfrep['phase'] == 'Menstrual') &
    (hormones_selfrep['phase'].shift(1) != 'Menstrual') &
    (hormones_selfrep['id'] == hormones_selfrep['id'].shift(1))  # same user as previous row
)

# Step 1: Calculate correctly ONLY for menstrual rows
hormones_selfrep['day_in_period'] = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Menstrual']
    .groupby(['id', 'period_id'])
    .cumcount() + 1
)

# Step 2: Fill ALL non-menstrual rows with 0 explicitly
hormones_selfrep['day_in_period'] = (
    hormones_selfrep['day_in_period']
    .fillna(0)
    .astype(int)
)

# Verify — max should now be 15
print(hormones_selfrep.groupby('id')['day_in_period'].max()
      .sort_values(ascending=False).head(10))

# Also verify menstrual rows still look correct
print(hormones_selfrep[hormones_selfrep['phase'] == 'Menstrual']
      ['day_in_period'].value_counts().sort_index())

# Calculate period_id per user (resets for each user)
hormones_selfrep['period_id'] = (
    hormones_selfrep
    .groupby('id')['period_start']
    .cumsum()
)

# Verify
print(hormones_selfrep.groupby('id')['day_in_period'].max().sort_values(ascending=False).head(10))

id
47    15
40    11
6      9
32     9
13     8
22     8
12     8
20     8
27     8
8      8
Name: day_in_period, dtype: int64
day_in_period
1     192
2     188
3     179
4     168
5     142
6     106
7      61
8      27
9       6
10      3
11      3
12      1
13      1
14      1
15      1
Name: count, dtype: int64
id
47    15
40    11
6      9
32     9
13     8
22     8
12     8
20     8
27     8
8      8
Name: day_in_period, dtype: int64


In [72]:
# Check what phase sequence looks like for user 43
user43 = hormones_selfrep[hormones_selfrep['id'] == 43][['day_in_study', 'phase', 'period_start', 'period_id', 'day_in_period']]
print(user43.head(50))

# How many distinct periods does user 43 have?
print(hormones_selfrep[hormones_selfrep['id'] == 43]['period_id'].value_counts())

# Check if phase is actually cycling for user 43
print(hormones_selfrep[hormones_selfrep['id'] == 43]['phase'].value_counts())

      day_in_study       phase  period_start  period_id  day_in_period
4589             1  Follicular         False          0              0
4590             2  Follicular         False          0              0
4591             3  Follicular         False          0              0
4592             4  Follicular         False          0              0
4593             5  Follicular         False          0              0
4594             6  Follicular         False          0              0
4595             7  Follicular         False          0              0
4596             8  Follicular         False          0              0
4597             9  Follicular         False          0              0
4598            10  Follicular         False          0              0
4599            11  Follicular         False          0              0
4600            12   Fertility         False          0              0
4601            13   Fertility         False          0              0
4602  

In [13]:
hormones_selfrep.groupby('id')['period_start'].sum()

id
1     3
2     3
3     3
4     4
6     3
7     4
8     3
9     6
10    6
11    3
12    7
13    6
14    6
15    4
16    2
18    7
19    3
20    8
22    6
23    3
24    2
26    9
27    6
29    3
30    4
32    3
33    6
34    3
37    4
38    7
39    3
40    2
41    7
42    8
43    3
44    4
45    3
46    3
47    6
48    5
49    2
50    9
Name: period_start, dtype: int64

In [75]:
# Step 1: Build day flow profile from existing non-null data
population_mode = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Menstrual']
    ['flow_volume_numeric']
    .mode()[0]
)

day_flow_profile = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Menstrual']
    .groupby('day_in_period')['flow_volume_numeric']
    .median()
)

print("Day flow profile:")
print(day_flow_profile)
print(f"\nPopulation mode (fallback): {population_mode}")

Day flow profile:
day_in_period
1     0.00
2     3.00
3     3.50
4     3.00
5     2.50
6     2.50
7     2.00
8     2.00
9     2.50
10    3.00
11    2.75
12    2.50
13     NaN
14    2.00
15     NaN
Name: flow_volume_numeric, dtype: float64

Population mode (fallback): 0.0


In [83]:
# Step 2: Define function
def impute_with_day_profile(row):
    if pd.notna(row['flow_volume_numeric']):
        return row['flow_volume_numeric']
    if row['phase'] == 'Menstrual':
        return day_flow_profile.get(row['day_in_period'], population_mode)
    return 0

# Step 3: Apply
hormones_selfrep['flow_volume_imputed'] = hormones_selfrep.apply(
    impute_with_day_profile, axis=1
)

# Step 4: Verify
print("\nNulls remaining:", hormones_selfrep['flow_volume_imputed'].isna().sum())
print("\nImputed distribution:")
print(hormones_selfrep['flow_volume_imputed'].value_counts().sort_index())

# Step 5: Sanity check - no non-zero values outside menstrual phase
non_menstrual_nonzero = hormones_selfrep[
    (hormones_selfrep['phase'] != 'Menstrual') & 
    (hormones_selfrep['flow_volume_imputed'] > 0)
]
print(f"\nNon-menstrual rows with flow > 0: {len(non_menstrual_nonzero)}")
# This should be 0


Nulls remaining: 2

Imputed distribution:
flow_volume_imputed
0.00    4537
1.00     198
2.00     187
2.50     217
2.75       1
3.00     294
3.50     142
4.00      56
5.00      25
Name: count, dtype: int64

Non-menstrual rows with flow > 0: 245


In [78]:
# Identify the 2 null rows
null_rows = hormones_selfrep[hormones_selfrep['flow_volume_imputed'].isna()]
print(null_rows[['id', 'day_in_study', 'phase', 'day_in_period', 'flow_volume_numeric', 'flow_color', 'is_bleeding']])

      id  day_in_study      phase  day_in_period  flow_volume_numeric  \
5088  47            44  Menstrual             13                  NaN   
5090  47            46  Menstrual             15                  NaN   

     flow_color  is_bleeding  
5088        NaN            1  
5090        NaN            1  


In [79]:
# Fill remaining 2 nulls with population mode
hormones_selfrep['flow_volume_imputed'] = (
    hormones_selfrep['flow_volume_imputed']
    .fillna(population_mode)
)

print("Nulls remaining:", hormones_selfrep['flow_volume_imputed'].isna().sum())
# Should be 0

Nulls remaining: 0


In [81]:
# Find where 2.75 came from
print(hormones_selfrep[hormones_selfrep['flow_volume_imputed'] == 2.75]
      [['id', 'day_in_study', 'phase', 'day_in_period', 'flow_volume_numeric']])

      id  day_in_study      phase  day_in_period  flow_volume_numeric
5086  47            42  Menstrual             11                  NaN


In [82]:
hormones_selfrep

,id,study_interval,is_weekend,day_in_study,phase,lh,estrogen,pdg,flow_volume,flow_color,...,is_bleeding,flow_grp_Abnormal,flow_grp_Fresh,flow_grp_No Flow,flow_grp_Old,flow_volume_numeric,period_start,period_id,day_in_period,flow_volume_imputed
0,1,2022,True,1,Follicular,2.9,94.2,NaN,Not at all,Not at all,...,0,False,False,True,False,0.0,False,0,0,0.0
1,1,2022,False,2,Follicular,1.2,226.3,NaN,Not at all,Not at all,...,0,False,False,True,False,0.0,False,0,0,0.0
2,1,2022,False,3,Follicular,3.5,276.8,NaN,Not at all,Not at all,...,0,False,False,True,False,0.0,False,0,0,0.0
3,1,2022,False,4,Fertility,1.8,322.1,NaN,Not at all,Not at all,...,0,False,False,True,False,0.0,False,0,0,0.0
4,1,2022,False,5,Fertility,4.6,244.9,NaN,Not at all,Not at all,...,0,False,False,True,False,0.0,False,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5654,50,2024,False,947,Luteal,4.6,70.7,2.8,NaN,NaN,...,0,False,False,False,False,NaN,False,8,0,0.0
5655,50,2024,False,948,Luteal,5.8,87.0,7.0,NaN,NaN,...,0,False,False,False,False,NaN,False,8,0,0.0
5656,50,2024,False,949,Luteal,NaN,NaN,NaN,NaN,NaN,...,0,False,False,False,False,NaN,False,8,0,0.0
5657,50,2024,False,950,Menstrual,NaN,NaN,NaN,NaN,NaN,...,1,False,False,False,False,NaN,True,9,1,0.0


In [84]:
valid_values = [0, 1, 2, 2.5, 3, 3.5, 4, 5]

def round_to_valid(val):
    return min(valid_values, key=lambda x: abs(x - val))

hormones_selfrep['flow_volume_imputed'] = (
    hormones_selfrep['flow_volume_imputed']
    .apply(round_to_valid)
)

print(hormones_selfrep['flow_volume_imputed'].value_counts().sort_index())

flow_volume_imputed
0.0    4539
1.0     198
2.0     187
2.5     218
3.0     294
3.5     142
4.0      56
5.0      25
Name: count, dtype: int64


In [86]:
hormones_selfrep

,id,study_interval,is_weekend,day_in_study,phase,lh,estrogen,pdg,flow_volume,flow_color,...,is_bleeding,flow_grp_Abnormal,flow_grp_Fresh,flow_grp_No Flow,flow_grp_Old,flow_volume_numeric,period_start,period_id,day_in_period,flow_volume_imputed
0,1,2022,True,1,Follicular,2.9,94.2,NaN,Not at all,Not at all,...,0,False,False,True,False,0.0,False,0,0,0.0
1,1,2022,False,2,Follicular,1.2,226.3,NaN,Not at all,Not at all,...,0,False,False,True,False,0.0,False,0,0,0.0
2,1,2022,False,3,Follicular,3.5,276.8,NaN,Not at all,Not at all,...,0,False,False,True,False,0.0,False,0,0,0.0
3,1,2022,False,4,Fertility,1.8,322.1,NaN,Not at all,Not at all,...,0,False,False,True,False,0.0,False,0,0,0.0
4,1,2022,False,5,Fertility,4.6,244.9,NaN,Not at all,Not at all,...,0,False,False,True,False,0.0,False,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5654,50,2024,False,947,Luteal,4.6,70.7,2.8,NaN,NaN,...,0,False,False,False,False,NaN,False,8,0,0.0
5655,50,2024,False,948,Luteal,5.8,87.0,7.0,NaN,NaN,...,0,False,False,False,False,NaN,False,8,0,0.0
5656,50,2024,False,949,Luteal,NaN,NaN,NaN,NaN,NaN,...,0,False,False,False,False,NaN,False,8,0,0.0
5657,50,2024,False,950,Menstrual,NaN,NaN,NaN,NaN,NaN,...,1,False,False,False,False,NaN,True,9,1,0.0


In [90]:
severity_map = {
    "Not at all": 0,
    "Very Low/Little": 1,
    "Very Low"
    "Low": 2,
    "Moderate": 3,
    "High": 4,
    "Very High": 5,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5
}

sym_cols = [
    'appetite', 'exerciselevel', 'headaches', 'cramps', 'sorebreasts',
    'fatigue', 'sleepissue', 'moodswing', 'stress', 'foodcravings',
    'indigestion', 'bloating'
]

for col in sym_cols:
    hormones_selfrep[col] = hormones_selfrep[col].map(severity_map)

In [92]:
# Check PDG and LH distributions
print("PDG stats:")
print(hormones_selfrep['pdg'].describe())

print("\nLH stats:")
print(hormones_selfrep['lh'].describe())

# Check PDG by phase - should be high in Luteal, low elsewhere
print("\nPDG median by phase:")
print(hormones_selfrep.groupby('phase')['pdg'].median())

# Check how often PDG and LH are both present
both_present = hormones_selfrep[['pdg', 'lh']].dropna()
print(f"\nRows where both present: {len(both_present)}")

PDG stats:
count    1864.000000
mean        6.229828
std         7.112508
min         1.000000
25%         1.500000
50%         3.400000
75%         7.700000
max        30.000000
Name: pdg, dtype: float64

LH stats:
count    5339.000000
mean        6.332609
std         7.888081
min         0.000000
25%         2.800000
50%         4.400000
75%         6.900000
max       185.600000
Name: lh, dtype: float64

PDG median by phase:
phase
Fertility     3.0
Follicular    2.0
Luteal        7.6
Menstrual     2.1
Name: pdg, dtype: float64

Rows where both present: 1864


In [93]:
# PDG threshold for confirmed ovulation
PDG_THRESHOLD = 5.0

# Get per-user stats from existing data
user_pdg_stats = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Luteal']
    .groupby('id')['pdg']
    .agg(['median', 'mean'])
)

population_luteal_pdg = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Luteal']
    ['pdg'].median()
)

population_follicular_pdg = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Follicular']
    ['pdg'].median()
)

print(f"Population Luteal PDG median: {population_luteal_pdg}")
print(f"Population Follicular PDG median: {population_follicular_pdg}")

Population Luteal PDG median: 7.6
Population Follicular PDG median: 2.0


In [94]:
# LH surge = LH value significantly above that user's baseline
# Typically defined as >2x baseline

user_lh_baseline = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Follicular']
    .groupby('id')['lh']
    .median()
)

def has_lh_surge(row):
    baseline = user_lh_baseline.get(row['id'], None)
    if baseline is None or pd.isna(row['lh']):
        return False
    return row['lh'] > (2 * baseline)

hormones_selfrep['lh_surge'] = hormones_selfrep.apply(has_lh_surge, axis=1)

print("LH surge days detected:")
print(hormones_selfrep['lh_surge'].value_counts())

LH surge days detected:
lh_surge
False    4991
True      668
Name: count, dtype: int64


In [95]:
def impute_pdg(row):
    # Keep original if exists
    if pd.notna(row['pdg']):
        return row['pdg']
    
    user_median = user_pdg_stats.get(row['id'], {}).get('median', None)
    
    # Luteal phase — PDG should be above threshold
    if row['phase'] == 'Luteal':
        return user_median if pd.notna(user_median) else population_luteal_pdg
    
    # Fertility phase — PDG starts rising after LH surge
    if row['phase'] == 'Fertility':
        if row['lh_surge']:
            # Just after ovulation, PDG beginning to rise
            return PDG_THRESHOLD * 0.8  # just below or at threshold
        return population_follicular_pdg  # pre-ovulation, still low
    
    # Follicular and Menstrual — PDG should be low
    if row['phase'] in ['Follicular', 'Menstrual']:
        return population_follicular_pdg

    return population_follicular_pdg  # safe default

hormones_selfrep['pdg_imputed'] = hormones_selfrep.apply(impute_pdg, axis=1)

# Verify
print("Nulls remaining:", hormones_selfrep['pdg_imputed'].isna().sum())
print("\nPDG imputed median by phase:")
print(hormones_selfrep.groupby('phase')['pdg_imputed'].median())

Nulls remaining: 0

PDG imputed median by phase:
phase
Fertility     2.0
Follicular    2.0
Luteal        7.6
Menstrual     2.0
Name: pdg_imputed, dtype: float64


In [ ]:
symptom_cols = [
    'exerciselevel', 'appetite', 'sleepissue', 'foodcravings', 
    'indigestion', 'stress', 'bloating', 'moodswing', 'fatigue', 
    'headaches', 'sorebreasts', 'cramps'
]

# Check data types and unique values
for col in symptom_cols:
    print(f"\n{col}:")
    print(hormones_selfrep[col].value_counts(dropna=False).head(6))

In [96]:
hormones_selfrep.to_csv('hormones_selfrep.csv', index=False)

In [24]:
cycles_per_person = (
    hormones_selfrep
    .groupby('id')['period_id']
    .nunique()
)
cycles_per_person

id
1      4
2      4
3      4
4      4
6      4
7      4
8      4
9      7
10     7
11     4
12     7
13     7
14     6
15     4
16     3
18     8
19     4
20     8
22     7
23     4
24     3
26    10
27     7
29     3
30     5
32     4
33     7
34     4
37     5
38     8
39     4
40     3
41     8
42     8
43     4
44     4
45     4
46     4
47     6
48     6
49     2
50    10
Name: period_id, dtype: int64

In [25]:
cycles_per_person.value_counts().sort_index()

period_id
2      1
3      4
4     18
5      2
6      3
7      7
8      5
10     2
Name: count, dtype: int64

In [26]:
hormones_selfrep[hormones_selfrep['id'] == 49][
    ['day_in_study','phase','period_id','day_in_period']
]


,day_in_study,phase,period_id,day_in_period
5428,1,Menstrual,1,1
5429,2,Menstrual,1,2
5430,3,Menstrual,1,3
5431,4,Menstrual,1,4
5432,5,Follicular,1,5
5433,6,Follicular,1,6
5434,7,Follicular,1,7
5435,8,Follicular,1,8
5436,9,Follicular,1,9
5437,10,Follicular,1,10


In [27]:
hormones_selfrep[hormones_selfrep['id'] == 43][
    ['day_in_study','period_start','period_id','day_in_period']
]

,day_in_study,period_start,period_id,day_in_period
4589,1,False,0,1
4590,2,False,0,2
4591,3,False,0,3
4592,4,False,0,4
4593,5,False,0,5
...,...,...,...,...
4770,940,False,3,78
4771,941,False,3,79
4772,942,False,3,80
4773,943,False,3,81


In [ ]:
hormones_selfrep['id'].value_counts()

id
26    210
42    204
33    194
47    193
50    193
12    192
38    192
27    192
20    191
41    191
14    190
9     190
22    190
48    190
30    190
13    189
43    186
18    185
10    175
32    124
6      90
7      90
4      90
3      90
1      90
2      90
11     90
8      90
39     90
24     90
16     90
15     90
23     90
19     90
46     90
40     90
34     90
37     90
44     90
45     90
29     60
49     38
Name: count, dtype: int64

In [ ]:
hormones_selfrep.groupby('id').size().describe()

count     42.000000
mean     134.738095
std       53.421499
min       38.000000
25%       90.000000
50%       90.000000
75%      190.000000
max      210.000000
dtype: float64

In [ ]:
hormones_selfrep

In [ ]:
# Step 4: Build the day profile from existing non-null data
population_mode = hormones_selfrep[hormones_selfrep['phase'] == 'Menstrual']['flow_volume_numeric'].mode()[0]

day_flow_profile = (
    hormones_selfrep[hormones_selfrep['phase'] == 'Menstrual']
    .groupby('day_in_period')['flow_volume_numeric']
    .median()
)

In [ ]:
# Step 5: Define the function
def impute_with_day_profile(row):
    if pd.notna(row['flow_volume_numeric']):
        return row['flow_volume_numeric']
    if row['phase'] == 'Menstrual':
        return day_flow_profile.get(row['day_in_period'], population_mode)
    return 0

In [ ]:
# Step 6: Apply it
hormones_selfrep['flow_volume_imputed'] = hormones_selfrep.apply(impute_with_day_profile, axis=1)

# Step 7: Verify results
print("Before imputation - nulls:", hormones_selfrep['flow_volume_numeric'].isna().sum())
print("After imputation - nulls:", hormones_selfrep['flow_volume_imputed'].isna().sum())
print("\nImputed value distribution:")
print(hormones_selfrep['flow_volume_imputed'].value_counts().sort_index())

C:\Users\iqra khan\AppData\Local\Temp\ipykernel_3108\2892171871.py:16: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  hormones_selfrep['phase'] = hormones_selfrep['phase'].fillna(method='ffill')


Before imputation - nulls: 2470
After imputation - nulls: 2

Imputed value distribution:
flow_volume_imputed
0.00    4537
1.00     198
1.50      16
1.75       3
2.00     171
2.50     214
2.75       1
3.00     294
3.50     142
4.00      56
5.00      25
Name: count, dtype: int64


In [ ]:
hormones_selfrep['period_start'].unique()
hormones_selfrep['day_in_period'].unique()

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
       18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34,
       35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51,
       52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68,
       69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82])

In [ ]:
def check_unique_values(df):
    for column in df.columns:
        print("Column:", column, "\nvalue:", df[column].unique(), "\n")

In [ ]:
severity_map = {
    "Not at all": 0,
    "Very Low/Little": 1,
    "Very Low"
    "Low": 2,
    "Moderate": 3,
    "High": 4,
    "Very High": 5,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5
}

sym_cols = [
    'appetite', 'exerciselevel', 'headaches', 'cramps', 'sorebreasts',
    'fatigue', 'sleepissue', 'moodswing', 'stress', 'foodcravings',
    'indigestion', 'bloating'
]

for col in sym_cols:
    hormones_selfrep[col] = hormones_selfrep[col].map(severity_map)

In [ ]:
hormones_selfrep.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5659 entries, 0 to 5658
Data columns (total 32 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   5659 non-null   int64  
 1   study_interval       5659 non-null   int64  
 2   is_weekend           5659 non-null   bool   
 3   day_in_study         5659 non-null   int64  
 4   phase                5659 non-null   object 
 5   lh                   5339 non-null   float64
 6   estrogen             5338 non-null   float64
 7   pdg                  1864 non-null   float64
 8   flow_volume          3189 non-null   object 
 9   flow_color           3194 non-null   object 
 10  appetite             2421 non-null   float64
 11  exerciselevel        1504 non-null   float64
 12  headaches            2824 non-null   float64
 13  cramps               3032 non-null   float64
 14  sorebreasts          2977 non-null   float64
 15  fatigue              2779 non-null   f

In [ ]:
flow_vol_map = {
    'Not at all' : 0,
    'Spotting / Very Light' : 1,
    'Light': 2,
    'Somewhat Light': 3,
    'Moderate' : 4,
    'Somewhat Heavy' : 5,
    'Heavy'  : 6,
    'Very Heavy' : 7
    }

hormones_selfrep['flow_volume'] = hormones_selfrep['flow_volume'].map(flow_vol_map)

In [ ]:
hormones_selfrep.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5659 entries, 0 to 5658
Data columns (total 32 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   5659 non-null   int64  
 1   study_interval       5659 non-null   int64  
 2   is_weekend           5659 non-null   bool   
 3   day_in_study         5659 non-null   int64  
 4   phase                5659 non-null   object 
 5   lh                   5339 non-null   float64
 6   estrogen             5338 non-null   float64
 7   pdg                  1864 non-null   float64
 8   flow_volume          3189 non-null   float64
 9   flow_color           3194 non-null   object 
 10  appetite             2421 non-null   float64
 11  exerciselevel        1504 non-null   float64
 12  headaches            2824 non-null   float64
 13  cramps               3032 non-null   float64
 14  sorebreasts          2977 non-null   float64
 15  fatigue              2779 non-null   f

In [ ]:
hormones_selfrep

,id,study_interval,is_weekend,day_in_study,phase,lh,estrogen,pdg,flow_volume,flow_color,...,is_bleeding,flow_grp_Abnormal,flow_grp_Fresh,flow_grp_No Flow,flow_grp_Old,flow_volume_numeric,period_start,period_id,day_in_period,flow_volume_imputed
0,1,2022,True,1,Follicular,2.9,94.2,NaN,0.0,Not at all,...,0,False,False,True,False,0.0,False,0,1,0.0
1,1,2022,False,2,Follicular,1.2,226.3,NaN,0.0,Not at all,...,0,False,False,True,False,0.0,False,0,2,0.0
2,1,2022,False,3,Follicular,3.5,276.8,NaN,0.0,Not at all,...,0,False,False,True,False,0.0,False,0,3,0.0
3,1,2022,False,4,Fertility,1.8,322.1,NaN,0.0,Not at all,...,0,False,False,True,False,0.0,False,0,4,0.0
4,1,2022,False,5,Fertility,4.6,244.9,NaN,0.0,Not at all,...,0,False,False,True,False,0.0,False,0,5,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5654,50,2024,False,947,Luteal,4.6,70.7,2.8,NaN,NaN,...,0,False,False,False,False,NaN,False,190,32,0.0
5655,50,2024,False,948,Luteal,5.8,87.0,7.0,NaN,NaN,...,0,False,False,False,False,NaN,False,190,33,0.0
5656,50,2024,False,949,Luteal,NaN,NaN,NaN,NaN,NaN,...,0,False,False,False,False,NaN,False,190,34,0.0
5657,50,2024,False,950,Menstrual,NaN,NaN,NaN,NaN,NaN,...,1,False,False,False,False,NaN,True,191,1,0.0


In [ ]:
hormones_selfrep.to_csv('hormones_selfrep.csv', index=False)

In [ ]:
hormones_selfrep_clean = hormones_selfrep.drop(columns = ['study_interval', 'exerciselevel', 'flow_color'])

In [ ]:
hormones_selfrep_clean

,id,is_weekend,day_in_study,phase,lh,estrogen,pdg,flow_volume,appetite,headaches,...,is_bleeding,flow_grp_Abnormal,flow_grp_Fresh,flow_grp_No Flow,flow_grp_Old,flow_volume_numeric,period_start,period_id,day_in_period,flow_volume_imputed
0,1,True,1,Follicular,2.9,94.2,NaN,0.0,NaN,4.0,...,0,False,False,True,False,0.0,False,0,1,0.0
1,1,False,2,Follicular,1.2,226.3,NaN,0.0,NaN,5.0,...,0,False,False,True,False,0.0,False,0,2,0.0
2,1,False,3,Follicular,3.5,276.8,NaN,0.0,NaN,4.0,...,0,False,False,True,False,0.0,False,0,3,0.0
3,1,False,4,Fertility,1.8,322.1,NaN,0.0,NaN,1.0,...,0,False,False,True,False,0.0,False,0,4,0.0
4,1,False,5,Fertility,4.6,244.9,NaN,0.0,NaN,1.0,...,0,False,False,True,False,0.0,False,0,5,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5654,50,False,947,Luteal,4.6,70.7,2.8,NaN,NaN,NaN,...,0,False,False,False,False,NaN,False,190,32,0.0
5655,50,False,948,Luteal,5.8,87.0,7.0,NaN,NaN,NaN,...,0,False,False,False,False,NaN,False,190,33,0.0
5656,50,False,949,Luteal,NaN,NaN,NaN,NaN,NaN,NaN,...,0,False,False,False,False,NaN,False,190,34,0.0
5657,50,False,950,Menstrual,NaN,NaN,NaN,NaN,NaN,NaN,...,1,False,False,False,False,NaN,True,191,1,0.0


In [ ]:
hormones_selfrep_clean.isnull().mean().sort_values(ascending=False)


pdg                    0.670613
appetite               0.572186
sleepissue             0.547270
foodcravings           0.525535
indigestion            0.512988
stress                 0.512281
bloating               0.510514
moodswing              0.509454
fatigue                0.508924
headaches              0.500972
sorebreasts            0.473935
cramps                 0.464216
flow_volume            0.436473
flow_volume_numeric    0.436473
estrogen               0.056724
lh                     0.056547
flow_volume_imputed    0.000353
id                     0.000000
is_weekend             0.000000
day_in_study           0.000000
phase                  0.000000
flow_grp_Abnormal      0.000000
is_bleeding            0.000000
flow_grp_No Flow       0.000000
flow_grp_Fresh         0.000000
flow_grp_Old           0.000000
period_start           0.000000
period_id              0.000000
day_in_period          0.000000
dtype: float64

In [ ]:
symptom_cols = [
    "headaches","cramps","sorebreasts","fatigue",
    "sleepissue","moodswing","stress",
    "foodcravings","indigestion","bloating", 
    "appetite", 'flow_volume'
]

hormones_selfrep_clean[symptom_cols] = hormones_selfrep_clean[symptom_cols].fillna(0)


In [ ]:
hormones_selfrep_clean['lh'] = hormones_selfrep_clean["lh"].fillna(hormones_selfrep_clean["lh"].median())
hormones_selfrep_clean['estrogen'] = hormones_selfrep_clean["estrogen"].fillna(hormones_selfrep_clean["estrogen"].median())

In [ ]:
# Convert bool to int
bool_cols = hormones_selfrep_clean.select_dtypes(include='bool').columns
hormones_selfrep_clean[bool_cols] = hormones_selfrep_clean[bool_cols].astype(int)

# Drop one flow group column
hormones_selfrep_clean = hormones_selfrep_clean.drop(columns=["flow_grp_No Flow"])

hormones_selfrep_clean

,id,is_weekend,day_in_study,phase,lh,estrogen,pdg,flow_volume,appetite,headaches,...,bloating,is_bleeding,flow_grp_Abnormal,flow_grp_Fresh,flow_grp_Old,flow_volume_numeric,period_start,period_id,day_in_period,flow_volume_imputed
0,1,1,1,Follicular,2.9,94.20,NaN,0.0,0.0,4.0,...,1.0,0,0,0,0,0.0,0,0,1,0.0
1,1,0,2,Follicular,1.2,226.30,NaN,0.0,0.0,5.0,...,1.0,0,0,0,0,0.0,0,0,2,0.0
2,1,0,3,Follicular,3.5,276.80,NaN,0.0,0.0,4.0,...,1.0,0,0,0,0,0.0,0,0,3,0.0
3,1,0,4,Fertility,1.8,322.10,NaN,0.0,0.0,1.0,...,1.0,0,0,0,0,0.0,0,0,4,0.0
4,1,0,5,Fertility,4.6,244.90,NaN,0.0,0.0,1.0,...,1.0,0,0,0,0,0.0,0,0,5,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5654,50,0,947,Luteal,4.6,70.70,2.8,0.0,0.0,0.0,...,0.0,0,0,0,0,NaN,0,190,32,0.0
5655,50,0,948,Luteal,5.8,87.00,7.0,0.0,0.0,0.0,...,0.0,0,0,0,0,NaN,0,190,33,0.0
5656,50,0,949,Luteal,4.4,100.05,NaN,0.0,0.0,0.0,...,0.0,0,0,0,0,NaN,0,190,34,0.0
5657,50,0,950,Menstrual,4.4,100.05,NaN,0.0,0.0,0.0,...,0.0,1,0,0,0,NaN,1,191,1,0.0


In [ ]:
hormones_selfrep_clean = hormones_selfrep_clean.sort_values(["id", "day_in_study"])

hormones_selfrep_clean["prev_bleeding"] = hormones_selfrep_clean.groupby("id")["is_bleeding"].shift(1)

hormones_selfrep_clean["period_start"] = (
    (hormones_selfrep_clean["is_bleeding"] == 1) &
    (hormones_selfrep_clean["prev_bleeding"] == 0)
).astype(int)


In [ ]:
hormones_selfrep_clean

,id,is_weekend,day_in_study,phase,lh,estrogen,pdg,flow_volume,appetite,headaches,...,is_bleeding,flow_grp_Abnormal,flow_grp_Fresh,flow_grp_Old,flow_volume_numeric,period_start,period_id,day_in_period,flow_volume_imputed,prev_bleeding
0,1,1,1,Follicular,2.9,94.20,NaN,0.0,0.0,4.0,...,0,0,0,0,0.0,0,0,1,0.0,NaN
1,1,0,2,Follicular,1.2,226.30,NaN,0.0,0.0,5.0,...,0,0,0,0,0.0,0,0,2,0.0,0.0
2,1,0,3,Follicular,3.5,276.80,NaN,0.0,0.0,4.0,...,0,0,0,0,0.0,0,0,3,0.0,0.0
3,1,0,4,Fertility,1.8,322.10,NaN,0.0,0.0,1.0,...,0,0,0,0,0.0,0,0,4,0.0,0.0
4,1,0,5,Fertility,4.6,244.90,NaN,0.0,0.0,1.0,...,0,0,0,0,0.0,0,0,5,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5654,50,0,947,Luteal,4.6,70.70,2.8,0.0,0.0,0.0,...,0,0,0,0,NaN,0,190,32,0.0,0.0
5655,50,0,948,Luteal,5.8,87.00,7.0,0.0,0.0,0.0,...,0,0,0,0,NaN,0,190,33,0.0,0.0
5656,50,0,949,Luteal,4.4,100.05,NaN,0.0,0.0,0.0,...,0,0,0,0,NaN,0,190,34,0.0,0.0
5657,50,0,950,Menstrual,4.4,100.05,NaN,0.0,0.0,0.0,...,1,0,0,0,NaN,1,191,1,0.0,0.0


In [ ]:
hormones_selfrep_clean["target_next_day"] = hormones_selfrep_clean.groupby("id")["period_start"].shift(-1)
hormones_selfrep_clean["target_next_day"] = hormones_selfrep_clean["target_next_day"].fillna(0)


In [ ]:
hormones_selfrep_clean["days_since_last_period"] = (
    hormones_selfrep_clean.groupby("id")["period_start"]
      .transform(lambda x: x.cumsum())
)


In [ ]:
drop_cols = [
    "period_start",   # derived internal
    "prev_bleeding"
]

hormones_selfrep_clean = hormones_selfrep_clean.drop(columns=drop_cols)


In [ ]:
hormones_selfrep_clean.describe()

,id,is_weekend,day_in_study,lh,estrogen,pdg,flow_volume,appetite,headaches,cramps,...,is_bleeding,flow_grp_Abnormal,flow_grp_Fresh,flow_grp_Old,flow_volume_numeric,period_id,day_in_period,flow_volume_imputed,target_next_day,days_since_last_period
count,5659.000000,5659.000000,5659.000000,5659.000000,5659.000000,1864.000000,5659.000000,5659.000000,5659.000000,5659.000000,...,5659.000000,5659.000000,5659.000000,5659.000000,3189.000000,5659.000000,5659.000000,5657.000000,5659.000000,5659.000000
mean,26.545503,0.278141,347.474289,6.223326,127.346289,6.229828,0.387878,1.415268,0.682276,0.486305,...,0.190670,0.041173,0.046475,0.056547,0.544998,96.643753,16.956529,0.501149,0.031984,2.522707
std,14.618883,0.448123,416.521099,7.674766,99.153877,7.112508,1.190840,1.676471,1.265325,1.046269,...,0.392864,0.198709,0.210529,0.230996,1.129172,54.456958,11.671867,1.093086,0.175974,1.923687
min,1.000000,0.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000
25%,13.000000,0.000000,34.000000,2.900000,68.900000,1.500000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,50.000000,8.000000,0.000000,0.000000,1.000000
50%,26.000000,0.000000,69.000000,4.400000,100.050000,3.400000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,99.000000,16.000000,0.000000,0.000000,2.000000
75%,41.000000,1.000000,893.000000,6.650000,151.100000,7.700000,0.000000,3.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,144.000000,24.000000,0.000000,0.000000,4.000000
max,50.000000,1.000000,1004.000000,185.600000,640.000000,30.000000,7.000000,5.000000,5.000000,5.000000,...,1.000000,1.000000,1.000000,1.000000,5.000000,191.000000,82.000000,5.000000,1.000000,9.000000


In [ ]:
hormones_selfrep_clean.to_csv('hormones_selfrep_clean.csv', index = False)

In [ ]:
# phase, pdg, id, 